---
title: "Exercise 5. MAGMA Gene-Set Analysis"
subtitle: "Test whether tissue-expression profiles are enriched for ADHD gene associations"
format:
  html:
    toc: true
    toc-depth: 3
    number-sections: true
jupyter: bash
---

# Overview

This notebook performs the final MAGMA step in the teaching pipeline: testing whether genes highly expressed in specific tissues are more associated with ADHD than expected by chance. Here we use the GTEx v8 average expression matrix as a gene-level covariate file.

::: {.callout-note}
Notebook 4 must have produced `output/magma/adhd_demo.genes.raw` and optionally `output/magma/adhd_full.genes.raw` before you run this notebook.
:::

::: {.callout-note}
## Learning goals
By the end of this notebook, you should be able to:
- explain what MAGMA gene-set analysis tests and how it differs from gene analysis
- use a tissue-expression covariate file as input to MAGMA
- interpret enrichment results as association between gene-level signal and tissue expression profiles
:::

::: {.callout-note}
## Big-picture questions

1. What does it mean biologically if brain tissues are enriched in this analysis?
2. Why is tissue enrichment still an indirect result rather than proof of causal cell types?
3. Why do we need multiple-testing correction across tissues?
:::

In [1]:
OUT_DIR=output/magma
GTEX=reference_data/gtex_v8_ts_avg_log2TPM.entrez.dedup.txt
RUN_MODE=both

mkdir -p ${OUT_DIR}

N_COLS=$(head -1 ${GTEX} | tr '\t' '\n' | wc -l)
printf 'GTEx columns detected: %s\n' "${N_COLS}"

GTEx columns detected: 56


# What MAGMA tests here

In this competitive analysis, MAGMA asks whether genes with higher values in a tissue-expression column tend to have stronger gene-level association statistics. That is different from simply asking whether those genes are highly expressed.

::: {.callout-warning}
A significant enrichment does not prove that the tissue is the unique site of action. Expression levels are correlated across tissues, and the test operates on average profiles rather than cell-state-specific mechanisms.
:::

In [2]:
run_geneset() {
  local label=$1
  local genes_raw=$2
  local out_prefix=$3

  echo
  echo '============================================================'
  echo "Running MAGMA gene-set analysis: ${label}"
  echo '============================================================'
  echo "Gene results file: ${genes_raw}"
  echo "GTEx covariate file: ${GTEX}"
  echo "Output prefix: ${out_prefix}"


magma \
    --gene-results ${genes_raw} \
    --gene-covar ${GTEX} \
    --out ${out_prefix}
}

In [3]:
run_geneset FULL ${OUT_DIR}/adhd_full.genes.raw ${OUT_DIR}/adhd_full


Running MAGMA gene-set analysis: FULL
Gene results file: output/magma/adhd_full.genes.raw
GTEx covariate file: reference_data/gtex_v8_ts_avg_log2TPM.entrez.dedup.txt
Output prefix: output/magma/adhd_full
Welcome to MAGMA v1.10 (linux/s)
Using flags:
	--gene-results output/magma/adhd_full.genes.raw
	--gene-covar reference_data/gtex_v8_ts_avg_log2TPM.entrez.dedup.txt
	--out output/magma/adhd_full

Start time is 13:21:17, Tuesday 11 Aug 2026

Reading file output/magma/adhd_full.genes.raw... 
	18112 genes read from file
Loading gene-level covariates...
Reading file reference_data/gtex_v8_ts_avg_log2TPM.entrez.dedup.txt... 
	detected 55 variables in file (using all)
	found 55 valid gene covariates, for 16966 genes defined in genotype data
Processing missing values...
	found 1146 genes not present in all input files: removing these from analysis
	16966 genes remaining in analysis
Preparing variables for analysis...
	truncating Z-scores 3 points below zero or 6 standard deviations above the 

In [4]:
run_geneset DEMO ${OUT_DIR}/adhd_demo.genes.raw ${OUT_DIR}/adhd_demo


Running MAGMA gene-set analysis: DEMO
Gene results file: output/magma/adhd_demo.genes.raw
GTEx covariate file: reference_data/gtex_v8_ts_avg_log2TPM.entrez.dedup.txt
Output prefix: output/magma/adhd_demo
Welcome to MAGMA v1.10 (linux/s)
Using flags:
	--gene-results output/magma/adhd_demo.genes.raw
	--gene-covar reference_data/gtex_v8_ts_avg_log2TPM.entrez.dedup.txt
	--out output/magma/adhd_demo

Start time is 13:21:24, Tuesday 11 Aug 2026

Reading file output/magma/adhd_demo.genes.raw... 
	67 genes read from file
Loading gene-level covariates...
Reading file reference_data/gtex_v8_ts_avg_log2TPM.entrez.dedup.txt... 
	detected 55 variables in file (using all)
	found 55 valid gene covariates, for 64 genes defined in genotype data
Processing missing values...
	found 3 genes not present in all input files: removing these from analysis
	64 genes remaining in analysis
Preparing variables for analysis...
	truncating Z-scores 3 points below zero or 6 standard deviations above the mean
	trunca

In [5]:
echo 'Top gene-set results by p-value:'
for gsa_out in ${OUT_DIR}/adhd_*.gsa.out; do
  [[ -f ${gsa_out} ]] || continue
  echo
  echo "Results: ${gsa_out}"
  awk 'NR==1 || /^VARIABLE/' ${gsa_out} | head -1
  grep -v '^#' ${gsa_out} | grep -v '^VARIABLE' | sort -k7 -g | head -10
done
echo
echo 'Bonferroni threshold for 54 tissues:'
echo 'scale=6; 0.05/54'

Top gene-set results by p-value:

Results: output/magma/adhd_demo.gsa.out
# MEAN_SAMPLE_SIZE = 225534
Heart_Left_Ventricle                COVAR      64     0.078207      0.11756     0.034072     0.026323 Heart_Left_Ventricle
Adipose_Visceral_Omentum            COVAR      64     0.061879      0.11905     0.027088     0.027014 Adipose_Visceral_Omentum
Heart_Atrial_Appendage              COVAR      64     0.073587      0.11412     0.034577     0.038708 Heart_Atrial_Appendage
Breast_Mammary_Tissue               COVAR      64     0.054154      0.10347     0.027844     0.057914 Breast_Mammary_Tissue
Adipose_Subcutaneous                COVAR      64      0.05088      0.10189     0.026249      0.05873 Adipose_Subcutaneous
Muscle_Skeletal                     COVAR      64     0.051683     0.095513     0.027293     0.064569 Muscle_Skeletal
Lung                                COVAR      64     0.052322     0.098929     0.028854     0.076302 Lung
Esophagus_Gastroesophageal_J...     COVAR      64  

# Discussion

The output `.gsa.out` file reports a regression-style enrichment result for each tissue. Focus on the sign and magnitude of the beta, but interpret the p-value in the context of multiple testing and the broader biology.

::: {.callout-tip}
## Reflection prompts
1. If multiple brain tissues are significant, how would you decide whether that reflects shared biology or correlated expression?
2. What follow-up analyses would you run to move from tissue enrichment toward cell-type or mechanism-level interpretation?
3. Why should the demo analysis be treated mainly as a teaching approximation rather than a biological conclusion?
:::